In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers datasets scikit-learn torch evaluate accelerate
!pip install sentencepiece  # needed for some tokenizers

# Core imports
import random
import numpy as np
import pandas as pd

# HuggingFace
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# runtime tracking
from tqdm import tqdm

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# device selection
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

Using device: cpu


In [ ]:
from datasets import Dataset

train_data_path = "data/training_data.csv"
train_label_path = "data/training_label.csv"
test_data_path = "data/testing_data.csv"
test_label_path = "data/testing_label.csv"


train_data = pd.read_csv(train_data_path)[0:100]
train_label = pd.read_csv(train_label_path)[0:100]
test_data = pd.read_csv(test_data_path)[0:100]
test_label = pd.read_csv(test_label_path)[0:100]

train_df = pd.concat([train_data, train_label], axis=1)
train_df["question_title"] = train_df["question_title"].fillna("")
train_df["question_content"] = train_df["question_content"].fillna("")
train_df["class_index"] = train_df["class_index"] - 1
test_df = pd.concat([test_data, test_label], axis=1)
test_df["question_title"] = test_df["question_title"].fillna("")
test_df["question_content"] = test_df["question_content"].fillna("")
test_df["class_index"] = test_df["class_index"] - 1

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


In [ ]:
print("============== Train data ==============")
print(train_data.head())
print(train_data.info())
print(train_data.describe())
print()
print("============== Train label ==============")
print(train_label.head())
print(train_label.info())
print(train_label.describe())
print()
print("============== Test data ==============")
print(test_data.head())
print(test_data.info())
print(test_data.describe())
print()
print("============== Test label ==============")
print(test_label.head())
print(test_label.info())
print(test_label.describe())


============== Train data ==============
                                      question_title  \
0  why doesn't an optical mouse work on a glass t...   
1       What is the best off-road motorcycle trail ?   
2             What is Trans Fat? How to reduce that?   
3                         How many planes Fedex has?   
4  In the san francisco bay area, does it make se...   

                                    question_content  
0                          or even on some surfaces?  
1                  long-distance trail throughout CA  
2  I heard that tras fat is bad for the body.  Wh...  
3  I heard that it is the largest airline in the ...  
4  the prices of rent and the price of buying doe...  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   question_title    100 non-null    object
 1   question_content  84 non-null     object
dtype

In [ ]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name, force_download=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=10,force_download=True)
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
# Tokenize the dataset
def tokenize(batch):
    tokenized = tokenizer(
        batch["question_title"],
        batch["question_content"],
        padding=True,
        truncation=True,
        max_length=128
    )
    if "class_index" in batch:
        tokenized["labels"] = batch["class_index"]
    return tokenized

# train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=["question_title", "question_content", "class_index"])
# test_dataset = test_dataset.map(tokenize, batched=True, remove_columns=["question_title", "question_content", "class_index"])


import os
from datasets import load_from_disk

if os.path.exists("tokenized_train"):
    tokenized_train = load_from_disk("tokenized_train")
    print("Tokenized train dataset loaded from disk")
else:
    tokenized_train = train_dataset.map(tokenize, batched=True, remove_columns=["question_title", "question_content", "class_index"])
    tokenized_train.save_to_disk("tokenized_train")

if os.path.exists("tokenized_test"):
    tokenized_test = load_from_disk("tokenized_test")
    print("Tokenized test dataset loaded from disk")
else:
    tokenized_test = test_dataset.map(tokenize, batched=True, remove_columns=["question_title", "question_content", "class_index"])
    tokenized_test.save_to_disk("tokenized_test")

split = tokenized_train.train_test_split(test_size=0.1, seed=42)
train_split = split["train"]
val_split = split["test"]

print("Train labels:", train_df["class_index"].unique())
print("Min:", train_df["class_index"].min(), "Max:", train_df["class_index"].max())



Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Train labels: [4 5 2 6 1 7 3 8 9 0]
Min: 0 Max: 9


In [ ]:
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir="model_results",
    per_device_train_batch_size=16,   # increase from 4
    gradient_accumulation_steps=2,    # 32x2=64 effective batch size
    gradient_checkpointing=False,     # turn off, hurts speed and not needed now
    fp16=True,
    num_train_epochs=5,               # add this explicitly
    learning_rate=2e-5,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    seed=42,
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_total_limit=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_split,
    eval_dataset=val_split,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)





In [ ]:
# For testing GPU memory allocation

# import torch
# batch = next(iter(trainer.get_train_dataloader()))
# print({k: v.shape for k, v in batch.items()})
# print("labels:", batch["labels"])
# print("input_ids min/max:", batch["input_ids"].min(), batch["input_ids"].max())

# # Try a manual forward pass to get the real error
# model.train()
# with torch.cuda.amp.autocast():
#     outputs = model(**batch)

In [ ]:
last_checkpoint = None
if os.path.isdir(model_name):
    checkpoints = [f for f in os.listdir(model_name) if f.startswith("checkpoint")]
    if checkpoints:
        last_checkpoint = os.path.join(model_name, sorted(checkpoints)[-1])
        print(f"Resuming from {last_checkpoint}")
    else:
        print("No checkpoints found, starting fresh")
else:
    print("Starting fresh")

trainer.train(resume_from_checkpoint=last_checkpoint)

Starting fresh


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=4.38133659362793, metrics={'train_runtime': 366.9394, 'train_samples_per_second': 1.226, 'train_steps_per_second': 0.041, 'total_flos': 14904708480000.0, 'train_loss': 4.38133659362793, 'epoch': 5.0})

In [16]:
# # TODO: Evaluate Model 1 on the test split
# test_results_1 = trainer.evaluate(eval_dataset=dataset_1_test)
# print("Model 1 validation results:", test_results_1)
# accuracy_1 = test_results_1['eval_accuracy']
# print(f"Model 1 validation accuracy: {accuracy_1:.4f}")

# from pathlib import Path
# # Load val.csv and tokenize
# val_df = pd.read_csv("val.csv")
# hf_val = Dataset.from_pandas(val_df, preserve_index=False)
# dataset_1_val = hf_val.map(tokenize_1, batched=True, remove_columns=["text"])
# # Predict
# pred_output = trainer.predict(dataset_1_val)
# pred_labels = np.argmax(pred_output.predictions, axis=-1)
# # Save in required format: single column "label"
# submission = pd.DataFrame({"label": pred_labels})
# out_dir = Path("outputs/model_1")
# out_dir.mkdir(parents=True, exist_ok=True)
# submission_path = out_dir / "val_predictions_labels_only.csv"
# submission.to_csv(submission_path, index=False)
# print(f"Saved: {submission_path.resolve()}")


# Test on test set
results = trainer.evaluate(eval_dataset=tokenized_test)
print("Results: ", results)

# Get predictions on test set
predictions = trainer.predict(tokenized_test)
logits = predictions.predictions
labels = predictions.label_ids

preds = np.argmax(logits, axis=-1)

class_names = ["Society & Culture",
               "Science & Mathematics",
               "Health",
               "Education & Reference",
               "Computers & Internet",
               "Sports",
               "Business & Finance",
               "Entertainment & Music",
               "Family & Relationships",
               "Politics & Government"]  # replace with your actual names

print(classification_report(labels, preds, target_names=class_names))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Results:  {'eval_loss': 2.26850962638855, 'eval_accuracy': 0.14, 'eval_f1': 0.0343859649122807, 'eval_runtime': 23.49, 'eval_samples_per_second': 4.257, 'eval_steps_per_second': 0.553, 'epoch': 5.0}


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


                        precision    recall  f1-score   support

     Society & Culture       0.00      0.00      0.00         6
 Science & Mathematics       0.14      1.00      0.25        14
                Health       0.00      0.00      0.00         7
 Education & Reference       0.00      0.00      0.00        10
  Computers & Internet       0.00      0.00      0.00        13
                Sports       0.00      0.00      0.00         5
    Business & Finance       0.00      0.00      0.00        22
 Entertainment & Music       0.00      0.00      0.00         8
Family & Relationships       0.00      0.00      0.00         5
 Politics & Government       0.00      0.00      0.00        10

              accuracy                           0.14       100
             macro avg       0.01      0.10      0.02       100
          weighted avg       0.02      0.14      0.03       100



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
